# Exit Hazard Model — Results

**Replaces:** Duration model (failed: mechanical collinearity with A_T)

**Model:** P(exit_i,t = 1) = σ(β₀ + β₁·A_T,t-lag + β₂·IL_t + β₃·log(age_i,t))

**Hypothesis:** β₁ > 0 → higher fee concentration drives LP exits → insurance demand

In [ ]:
from econometrics.data import DAILY_AT_MAP, IL_MAP, RAW_POSITIONS
from econometrics.ingest import build_exit_panel
from econometrics.hazard import (
    logit_mle,
    marginal_effect_at_means,
    exit_quartile_analysis,
    exit_lag_sensitivity,
)

LAG = 1
panel = build_exit_panel(RAW_POSITIONS, DAILY_AT_MAP, IL_MAP, lag_days=LAG)
print(f"Panel size: {len(panel)} position-day observations")
print(f"Exits: {sum(1 for r in panel if r.exited == 1)}")
print(f"Days: {len(set(r.day for r in panel))}")

## 1. Main Result

In [ ]:
result = logit_mle(panel)
print(f"{'':>20} {'Coeff':>10} {'MLE SE':>10} {'Cluster SE':>10} {'Cluster p':>10}")
print(f"{'\u03b2\u2081 (A_T)':>20} {result.beta_a_t:>10.4f} {result.se_a_t:>10.4f} {result.cluster_se_a_t:>10.4f} {result.cluster_p_value_a_t:>10.6f}")
print(f"{'\u03b2\u2082 (IL)':>20} {result.beta_il:>10.4f} {result.se_il:>10.4f} {result.cluster_se_il:>10.4f}")
print(f"{'\u03b2\u2083 (log age)':>20} {result.beta_log_age:>10.4f} {result.se_log_age:>10.4f} {result.cluster_se_log_age:>10.4f}")
print(f"{'\u03b2\u2080 (intercept)':>20} {result.beta_intercept:>10.4f}")
print(f"\nn={result.n_obs}  exits={result.n_exits}  clusters={result.n_clusters}")
print(f"LL={result.log_likelihood:.1f}  AIC={result.aic:.1f}  pseudo-R\u00b2={result.pseudo_r2:.4f}")
print(f"Mean P(exit)={result.mean_exit_prob:.4f}")

## 2. Economic Magnitude (Insurance Pricing)

In [ ]:
me = marginal_effect_at_means(result, delta_a_t=0.10)
print(f"Marginal effect dP(exit)/dA_T:  {me.marginal_effect:.6f}")
print(f"A 0.10 increase in A_T:")
print(f"  Exit probability increase:    {me.prob_increase:.6f}")
print(f"  Hours of fee revenue lost:    {me.hours_lost:.2f}")
print(f"  Implied insurance premium:    ${me.implied_premium_usd:.2f}")
print(f"  (Mean exit probability:       {me.mean_exit_prob:.4f})")
if result.beta_a_t > 0:
    print(f"\nINTERPRETATION: \u03b2\u2081 > 0 confirms insurance demand.")
    print(f"PLPs exit faster when fee concentration is high.")
else:
    print(f"\n\u03b2\u2081 <= 0 -- concentration does not drive exits in this sample.")

## 3. Dose-Response (A_T Quartiles)

In [ ]:
import matplotlib.pyplot as plt

quartiles = exit_quartile_analysis(panel)
qs = [q.quartile for q in quartiles]
exit_rates = [q.mean_blocklife_hours for q in quartiles]
ats = [q.mean_a_t for q in quartiles]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(qs, exit_rates, color=["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"], edgecolor="black")
ax.set_xlabel("A_T Quartile (lagged)")
ax.set_ylabel("Exit Rate (%)")
ax.set_title("Dose-Response: Fee Concentration vs Exit Probability")
ax.set_xticks(qs)
ax.set_xticklabels([f"Q{q}\n(A_T={a:.3f})" for q, a in zip(qs, ats)])

for bar, rate, n in zip(bars, exit_rates, [q.n_obs for q in quartiles]):
    ax.text(bar.get_x() + bar.get_width()/2, rate + 0.1, f"{rate:.2f}%\n(n={n})",
            ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 4. Lag Sensitivity

In [ ]:
rows = exit_lag_sensitivity(RAW_POSITIONS, DAILY_AT_MAP, IL_MAP)
print(f"{'Lag':>4} {'\u03b2\u2081':>10} {'Cluster SE':>12} {'p-value':>10} {'n':>8} {'Sig':>4}")
print("-" * 52)
for r in rows:
    sig = "***" if r.robust_p_value_a_t < 0.01 else "**" if r.robust_p_value_a_t < 0.05 else "*" if r.robust_p_value_a_t < 0.10 else ""
    print(f"{r.lag:>4} {r.beta_a_t:>10.4f} {r.robust_se_a_t:>12.4f} {r.robust_p_value_a_t:>10.4f} {r.n_obs:>8} {sig:>4}")